Copyright (c) 2025 Huawei Technologies Co., Ltd. This program is free software, you can redistribute it and/or modify it under the terms and conditions of CANN Open Software License Agreement Version 2.0 (the "License"). Please refer to the License for details. You may not use this file except in compliance with the License. THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE. See LICENSE in the root of the software repository for the full text of the License.

# YOLOv13 pyACL 免编译端到端推理

本案例由社区开发者贡献，是官方案例 [reference_practice/yolov13_offline_inference](../../../reference_practice/yolov13_offline_inference/README.md) 第 3 章 ACL 离线推理 C++ 程序（`yolov13_main`）的**免编译备选**。

在缺少 C++ 编译环境（如精简镜像未预装 g++/gcc-c++，CMake 报 `g++ is not a full path`）或希望快速验证推理结果的场景下，可用 pyACL（Python 版 ACL 接口）完成与 C++ 程序完全相同的推理流程。**本 Notebook 是端到端可独立执行的**：模型转换、输入预处理与推理后处理全部在下面的 Cell 中完成，不依赖官方案例的中间产物。

> 本案例不改变官方 C++ 推理路径，仅为其提供低成本替代。

## 学习目标

- 完成从权重下载、ONNX 导出、ATC 模型编译到输入预处理的端到端准备；
- 使用 pyACL 加载 OM 模型并执行推理，理解 `(84, 8400)` 输出布局的解析、class-wise NMS 与结果落盘；
- 使用官方案例的 `draw_boxes.py` 可视化检测结果，并与 C++ 程序的期望输出逐字比对。

## 环境要求

| 项目 | 要求 |
| --- | --- |
| 硬件 | 昇腾 Atlas 推理系列（已在 Atlas 300I Duo / 310P 实机验证） |
| CANN | 8.0.0（Toolkit，含 pyACL）及以上 |
| Python | 3.x（官方案例推荐 3.11），依赖 numpy |

## 1 环境检查

推理前先确认三件事：CANN 环境变量已生效（能 `import acl`）、numpy 可用、昇腾设备在线。本案例默认使用 0 号设备，多卡环境可在后面的 Cell 中通过变量切换。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# CANN 环境变量（若在终端已 source set_env.sh，可跳过本 Cell 的 source 步骤）
CANN_ROOT = Path(os.environ.get('ASCEND_HOME_PATH',
                                '/usr/local/Ascend/ascend-toolkit/latest'))
if (CANN_ROOT / 'set_env.sh').exists():
    lib64 = str(CANN_ROOT / 'aarch64-linux' / 'lib64')
    os.environ['LD_LIBRARY_PATH'] = lib64 + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import numpy as np
import acl
print('CANN_ROOT =', CANN_ROOT)
print('numpy', np.__version__, '/ pyACL import OK')
ret = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True)
print(ret.stdout if ret.returncode == 0 else 'npu-smi 不可用，请确认昇腾驱动与设备')

## 2 端到端准备：权重 → ONNX → OM → 输入 BIN

这一节把官方案例第 2 章的产物在本 Notebook 内部做出来，实现**独立执行**：

1. 拉取 YOLOv13 源码并安装依赖（YOLOv13 使用自定义网络模块，需从镜像仓库源码安装）；
2. 下载 `yolov13n.pt` 权重并导出固定输入 `1x3x640x640` 的 ONNX；
3. 自动识别 `soc_version` 后用 ATC 编译出 OM；
4. 下载官方测试图 `bus.jpg` 并用官方案例的 `scripts/preprocess.py` 生成 FP32 的 `bus.bin`。

下面的 Cell 有先后关系，后面的路径变量沿用前面的结果。

### 2.1 初始化目录与依赖

In [ ]:
CASE_DIR = Path.cwd()
TUTORIAL_DIR = CASE_DIR.parents[1] / 'reference_practice' / 'yolov13_offline_inference'  # 官方案例根目录
WORK_DIR = CASE_DIR / 'workspace'
MODEL_DIR = WORK_DIR / 'model'
DATA_DIR = WORK_DIR / 'data'
for d in (WORK_DIR, MODEL_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('案例目录:', CASE_DIR)
print('官方案例:', TUTORIAL_DIR, '存在' if TUTORIAL_DIR.exists() else '不存在，请确认仓库完整')

In [ ]:
# YOLOv13 需源码安装（普通 ultralytics 版本无法加载其自定义模块）
import subprocess, sys
YOLOV13_DIR = WORK_DIR / 'yolov13'
if not YOLOV13_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://gitcode.com/gh_mirrors/yo/yolov13.git', str(YOLOV13_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(YOLOV13_DIR)], check=True)
print('yolov13 依赖安装完成')

### 2.2 下载权重并导出 ONNX

固定 Shape 与后面的 ATC `--input_shape` 必须一致。权重下载与 ONNX 导出复用官方案例的 `scripts/download_weight.py`，保证与官方教程取到同一份产物。

In [ ]:
WEIGHT_PATH = WORK_DIR / 'yolov13n.pt'
ONNX_PATH = WORK_DIR / 'yolov13n.onnx'
if not WEIGHT_PATH.exists():
    subprocess.run([sys.executable, str(TUTORIAL_DIR / 'scripts' / 'download_weight.py'),
                    '--output', str(WEIGHT_PATH)], check=True)
if not ONNX_PATH.exists():
    from ultralytics import YOLO
    model = YOLO(str(WEIGHT_PATH))
    model.export(format='onnx', imgsz=640, simplify=True)  # 生成 1x3x640x640 固定输入
    ONNX_PATH.rename(ONNX_PATH) if Path(str(ONNX_PATH)).exists() else None
print('ONNX:', ONNX_PATH, ONNX_PATH.exists())

### 2.3 自动识别 `soc_version` 并用 ATC 编译 OM

按官方建议先用 `npu-smi info -t board` 读取芯片型号；取不到时回退为 `Ascend<设备名>`。`--output_type=FP32` 保证输出可直接按 `(84, 8400)` 的 FP32 布局解析。

In [ ]:
import re
npu_env = os.environ.copy()
npu_id, chip_id = os.environ.get('NPU_ID', '0'), os.environ.get('CHIP_ID', '0')
out = subprocess.run(['npu-smi', 'info', '-t', 'board', '-i', npu_id, '-c', chip_id],
                     capture_output=True, text=True, env=npu_env)
names = re.findall(r'(?:Chip Name|NPU Name)\s*:\s*(\S+)', out.stdout)
m = re.search(r'Ascend(\S+)', out.stdout)
if names:
    soc_version = '_'.join(names)
elif m:
    soc_version = 'Ascend' + m.group(1)
else:
    soc_version = 'Ascend310P'
print('soc_version =', soc_version)
OM_PREFIX = MODEL_DIR / 'yolov13'
if not Path(str(OM_PREFIX) + '.om').exists():
    subprocess.run(['atc', '--model=' + str(ONNX_PATH), '--framework=5',
                    '--output=' + str(OM_PREFIX), '--soc_version=' + soc_version,
                    '--input_shape=images:1,3,640,640', '--output_type=FP32'], check=True)
print('OM:', str(OM_PREFIX) + '.om', Path(str(OM_PREFIX) + '.om').exists())

### 2.4 下载测试图并生成 ACL 输入 BIN

预处理脚本做 letterbox 缩放、RGB 转换、归一化并转 NCHW，写出 FP32 的 `bus.bin`；脚本需在 `data` 目录执行（它扫描当前目录中的图片）。

In [ ]:
IMAGE_PATH = DATA_DIR / 'bus.jpg'
if not IMAGE_PATH.exists():
    subprocess.run(['wget', '-O', str(IMAGE_PATH),
                    'https://raw.gitcode.com/gh_mirrors/yo/yolov13/raw/main/bus.jpg'], check=True)
if not (DATA_DIR / 'bus.bin').exists():
    subprocess.run([sys.executable, str(TUTORIAL_DIR / 'scripts' / 'preprocess.py')],
                   cwd=DATA_DIR, check=True)
print('输入 BIN:', DATA_DIR / 'bus.bin', (DATA_DIR / 'bus.bin').exists())

## 3 pyACL 推理分步实现

推理代码按步骤拆分讲解，各步骤直接调用本案例 `scripts/infer_pyacl.py` 中已通过评审的实现（避免同一逻辑维护两份）。流程为：**acl 初始化 → 加载 OM → 构造输入/输出 Dataset → `aclmdlExecute` → 解析 `(84, 8400)` 输出 → class-wise NMS → 写 `_result.txt`**。

In [ ]:
# 导入本案例脚本（其内部已定义常量与全部函数）
sys.path.insert(0, str(CASE_DIR / 'scripts'))
import infer_pyacl as ip
print('脚本导入成功:', ip.__doc__.splitlines()[0])

### 3.1 初始化 ACL 运行时并加载模型

`acl.init` → `set_device` → `create_context`/`create_stream` 完成运行时初始化；`acl.mdl.load_from_file` 加载 OM 后用 `get_desc` 取模型描述，后续按描述查询输入/输出尺寸。若 `--model` 路径写错，`model_id` 保持 `None`，`finally` 中判空后再释放，避免 `UnboundLocalError` 掩盖真实错误。

In [ ]:
MODEL_PATH = str(OM_PREFIX) + '.om'
context, stream = ip.init_acl()
model_id, desc = ip.load_model(MODEL_PATH)
in_size = acl.mdl.get_input_size_by_index(desc, 0)
out_size = acl.mdl.get_output_size_by_index(desc, 0)
print(f'input {in_size}B, output {out_size}B（期望 {ip.NUM_ANCHORS * (ip.NUM_CLASSES + 4) * 4}B）')

### 3.2 构造输入/输出 Dataset 并执行推理

输入：读入 `bus.bin` 并校验大小与模型一致，经 `bytes_to_ptr` + `memcpy` H2D 拷入 Device；输出：malloc Device 内存挂到 Dataset。`acl.mdl.execute` 同步执行一次推理。

In [ ]:
with open(DATA_DIR / 'bus.bin', 'rb') as f:
    host_in = f.read()
in_ds, dbuf = ip.build_dataset_from_buffer(host_in, in_size, is_input=True)
out_ds, obuf = ip.build_dataset_from_buffer(None, out_size, is_input=False)
ip.check(ip.norm(acl.mdl.execute(model_id, in_ds, out_ds)), 'aclmdlExecute failed')
print('推理执行完成')

### 3.3 解析输出、NMS 与结果落盘

输出张量按 `(4 + 80, 8400)` 行布局解析：每列为一个 anchor，前 4 行是框参数，后 80 行是类别分数。取每列最大类别分数做置信度过滤（`>= conf`，与 C++ 一致），换算裁剪到 640×640；NMS 仅在**同类**框之间做 IoU 抑制（与 `sample_yolov13.cpp` 的 class-wise NMS 一致，IoU 分母含 1e-6 平滑项）。结果按 `label conf x1 y1 x2 y2` 写入 `bus_result.txt`，与 C++ 程序输出格式逐字一致。

In [ ]:
CONF, IOU = 0.25, 0.45  # 与 sample_yolov13.cpp 的 kConfThresh / kIouThresh 默认值一致
out = ip.fetch_output(obuf, out_size)
keep = ip.nms(ip.parse_yolo_output(out, CONF), IOU)
ip.report_and_save(keep, str(DATA_DIR / 'bus.bin'))

### 3.4 释放资源

Device 内存、Dataset、模型、上下文、流与设备按申请的逆序统一释放；pyACL 部分接口在不同 CANN 版本返回 `(ptr, ret)` 元组，脚本内 `norm()` 已做兼容。

In [ ]:
for ds in (in_ds, out_ds):
    acl.mdl.destroy_dataset(ds)
for buf in (dbuf, obuf):
    acl.rt.free(buf)
acl.mdl.unload(model_id)
acl.mdl.destroy_desc(desc)
acl.rt.destroy_context(context)
acl.rt.destroy_stream(stream)
acl.rt.reset_device(0)
acl.finalize()
print('资源已释放')

## 4 整脚本方式运行与结果可视化对比

除了分步执行，也可以直接运行本案例的整脚本（参数与官方案例 `run.sh` 命名一致），二者结果完全相同：

In [ ]:
# 分步执行后资源已释放，整脚本方式会独立完成一次全流程
subprocess.run([sys.executable, str(CASE_DIR / 'scripts' / 'infer_pyacl.py'),
                '--model=' + MODEL_PATH, '--input=' + str(DATA_DIR / 'bus.bin')], check=True)

结果文件 `bus_result.txt` 可直接交给官方案例的 `scripts/draw_boxes.py` 生成带检测框的图片，与 C++ 程序 `yolov13_main` 的期望输出（bus 0.74 + person×4，`recognized persons: 4, vehicles: 1`）逐字比对。

In [ ]:
from IPython.display import Image, display
# draw_boxes.py 按官方案例约定在 data 目录内查找 *_result.txt 并输出 bus_draw.jpg
subprocess.run([sys.executable, str(TUTORIAL_DIR / 'scripts' / 'draw_boxes.py')],
               cwd=DATA_DIR, check=True)
draw_path = DATA_DIR / 'bus_draw.jpg'
if draw_path.exists():
    display(Image(filename=str(draw_path)))
else:
    print('未生成可视化图片，请查看 draw_boxes.py 的输出目录约定')

## 5 常见问题

1. **`ImportError: cannot import name 'acl'`**：未安装 CANN Toolkit 或未 source `set_env.sh`；请先安装 Toolkit 并设置环境变量。
2. **`输入大小与模型不符`**：`--input` 指定的 BIN 不是由官方案例 `scripts/preprocess.py` 针对该 OM 模型生成；请重新执行 2.4 节。
3. **`acl.rt.set_device failed`**：无可用的昇腾设备或设备被占用；用 `npu-smi info` 确认设备状态。当前脚本固定使用 0 号设备。
4. **检出结果为空**：检查输入是否为正确的 `.bin`（而非原始图片）；确认 `--conf` 阈值未被设置过高。
5. **ATC 编译失败 `soc_version` 不合法**：2.3 节自动识别失败时会回退为 `Ascend310P`，若与实际芯片不符请手工设置环境变量 `SOC_VERSION` 或修改对应 Cell。
6. **与 C++ 结果不一致**：确认 `--conf`/`--iou` 未被改动（默认 0.25/0.45），且 OM 由 `--output_type=FP32` 编译；本案例解析阈值与 `sample_yolov13.cpp` 逐项对齐。

## 反馈

问题与建议欢迎在本仓库提 Issue 并 @ 贡献者，或在官方案例对应 Issue 中讨论。整脚本形态见 [scripts/infer_pyacl.py](scripts/infer_pyacl.py)，说明索引见 [README.md](README.md)。